# Global Conv1D Attention Challenger

This notebook independently applies the shared 24-month/H1-H12 Conv1D-attention contract to the v8 controlled panel. The canonical operational-baseline notebooks remain the source of truth for the full challenger leaderboard.

The neural model uses causal 3- and 6-month convolutions, two residual self-attention blocks, origin-safe known-future and spectral branches, ordered quantiles and five deterministic seeds. Attention weights are descriptive and are not causal effects.


In [1]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import Image, display
except ImportError:
    class Image:
        def __init__(self, filename): self.filename = filename
        def __repr__(self): return f"Image({self.filename})"
    def display(value): print(value)

ROOT = Path.cwd()
if ROOT.name != "v8_controlled_synthetic_validation":
    candidate = ROOT / "Ai miroservices/modeling/v8_controlled_synthetic_validation"
    ROOT = candidate if candidate.exists() else ROOT
OUT = ROOT / "outputs"
DATA = OUT / "data"
PLOTS = OUT / "plots"
EVAL = OUT / "evaluator"
summary = json.loads((OUT / "run_summary.json").read_text())


In [2]:
evaluator_summary=json.loads((EVAL/'evaluator_run_summary.json').read_text())
evaluator_board=pd.read_csv(EVAL/'model_leaderboard.csv')
seed=pd.read_csv(EVAL/'neural_seed_stability.csv')
display(pd.DataFrame([evaluator_summary]).T)
display(evaluator_board)
display(seed.groupby('split')[['WAPE','RMSE','epochs']].agg(['mean','std','min','max']))


,0
data_tier,CONTROLLED_SYNTHETIC_GROUND_TRUTH
purpose,Independent v8 replication of the shared neura...
history_months,24
horizon_months,12
seeds,"[17, 42, 101, 303, 707]"
production_decision_eligible,False
external_population_validity,UNVERIFIED
leaderboard,"[{'split': 'selection', 'model_name': 'CONV1D_..."


,split,model_name,rows,series,WAPE,MAE,RMSE,Bias,under_forecast_rate,selection_score
0,selection,CONV1D_ATTENTION_GLOBAL,1440,120,0.113978,948.123724,2056.156381,0.079827,0.659722,0.153892
1,selection,SEASONAL_NAIVE,1440,120,0.142392,1184.483134,2295.132716,0.075290,0.613889,0.180037
2,untouched_test,CONV1D_ATTENTION_GLOBAL,1440,120,0.098996,874.987305,1623.956717,0.031222,0.558333,0.114607
3,untouched_test,SEASONAL_NAIVE,1440,120,0.136170,1203.553026,2222.646504,0.058849,0.617361,0.165595


WAPE                                       RMSE  \
                    mean       std       min       max         mean   
split                                                                 
selection       0.126781  0.007966  0.117131  0.135242  2249.981250   
untouched_test  0.107008  0.005048  0.100953  0.113411  1766.519507   

                                                     epochs                     
                       std          min          max   mean        std min max  
split                                                                           
selection       269.983455  1962.203979  2638.824463   49.0   8.746428  40  63  
untouched_test   89.367445  1656.753052  1904.070801   53.0  18.027756  30  72

In [3]:
attention=pd.read_csv(EVAL/'attention_weights.csv')
occlusion=pd.read_csv(EVAL/'lag_occlusion_sensitivity.csv')
permutation=pd.read_csv(EVAL/'heldout_group_permutation.csv')
display(attention)
display(occlusion.sort_values('WAPE_increase',ascending=False).head(12))
display(permutation.sort_values('WAPE_increase',ascending=False))


,lag_position,mean_attention_weight,note
0,-24,0.046880,Attention weights are descriptive and are not ...
1,-23,0.041644,Attention weights are descriptive and are not ...
2,-22,0.041818,Attention weights are descriptive and are not ...
3,-21,0.038022,Attention weights are descriptive and are not ...
4,-20,0.038266,Attention weights are descriptive and are not ...
5,-19,0.038056,Attention weights are descriptive and are not ...
6,-18,0.037585,Attention weights are descriptive and are not ...
7,-17,0.038689,Attention weights are descriptive and are not ...
8,-16,0.037756,Attention weights are descriptive and are not ...
9,-15,0.044146,Attention weights are descriptive and are not ...


,lag_position,baseline_WAPE,occluded_WAPE,WAPE_increase,note
23,-1,0.113411,0.113657,0.000246,"Single-seed lag occlusion; association, not ca..."
10,-14,0.113411,0.113644,0.000233,"Single-seed lag occlusion; association, not ca..."
9,-15,0.113411,0.113617,0.000206,"Single-seed lag occlusion; association, not ca..."
11,-13,0.113411,0.113555,0.000144,"Single-seed lag occlusion; association, not ca..."
8,-16,0.113411,0.113549,0.000138,"Single-seed lag occlusion; association, not ca..."
1,-23,0.113411,0.113535,0.000124,"Single-seed lag occlusion; association, not ca..."
12,-12,0.113411,0.113512,0.000101,"Single-seed lag occlusion; association, not ca..."
16,-8,0.113411,0.113463,0.000052,"Single-seed lag occlusion; association, not ca..."
0,-24,0.113411,0.113462,0.000051,"Single-seed lag occlusion; association, not ca..."
2,-22,0.113411,0.113451,0.000040,"Single-seed lag occlusion; association, not ca..."


,feature_group,baseline_WAPE,permuted_WAPE,WAPE_increase,note
1,known_future,0.113411,0.160168,0.046757,"Held-out group permutation; association, not c..."
2,spectral_summary,0.113411,0.115055,0.001644,"Held-out group permutation; association, not c..."
0,past_exogenous,0.113411,0.113934,0.000523,"Held-out group permutation; association, not c..."
3,static_features,0.113411,0.113660,0.000249,"Held-out group permutation; association, not c..."


A neural win on generated data validates only this controlled experiment. External population validity remains `UNVERIFIED`.
